# 2.5 — Feature Engineering

Feature engineering means creating new columns from existing ones to give the model more useful signal.

> **Core idea:** Raw data rarely comes in the perfect shape for a model. A loan amount means nothing without income context. A birth year means nothing without knowing the current year. You help the model see the patterns.

### Techniques covered:
- **Math features** — ratios, differences, products
- **Binning** — converting continuous numbers into categories
- **Extracting info** — pulling useful data from dates and text
- **Interaction features** — combining two columns into one

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':        ['Ali', 'Bina', 'Charu', 'Dev', 'Eva'],
    'income':      [50000, 80000, 35000, 120000, 60000],
    'loan_amt':    [10000, 60000, 15000,  80000, 25000],
    'expenses':    [20000, 45000, 18000,  55000, 30000],
    'age':         [28, 45, 23, 52, 34],
    'joined_date': pd.to_datetime(['2021-03-15', '2019-07-22',
                                   '2023-01-10', '2018-11-05', '2022-06-18']),
    'defaulted':   [0, 1, 0, 0, 1]
})

df

## 1. Math Features — Ratios & Differences

Ratios beat raw numbers. A ₹60,000 loan means very different things for someone earning ₹80,000 vs ₹35,000.

In [ ]:
# debt-to-income ratio — how burdened is this person?
df['debt_ratio'] = df['loan_amt'] / df['income']

# disposable income — what's left after expenses?
df['disposable'] = df['income'] - df['expenses']

# savings rate — what fraction do they save?
df['savings_rate'] = df['disposable'] / df['income']

df[['name', 'income', 'loan_amt', 'expenses',
    'debt_ratio', 'disposable', 'savings_rate']].round(2)

## 2. Binning — Converting Numbers to Categories

Sometimes the exact number matters less than the range it falls in.

In [ ]:
# pd.cut — manual boundaries
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 50, 100],
    labels=['young', 'mid', 'senior']
)

df[['name', 'age', 'age_group']]

In [ ]:
# pd.qcut — equal frequency bins (each bin has same number of rows)
df['income_band'] = pd.qcut(
    df['income'], q=3, labels=['low', 'mid', 'high']
)

df[['name', 'income', 'income_band']]

## 3. Extracting Info from Datetime Columns

A single date column contains many hidden features — year, month, day of week, season, time since event etc.

In [ ]:
df['join_year']          = df['joined_date'].dt.year
df['join_month']         = df['joined_date'].dt.month
df['join_day_of_week']   = df['joined_date'].dt.dayofweek  # 0=Mon, 6=Sun
df['join_is_weekend']    = df['join_day_of_week'].isin([5, 6]).astype(int)
df['years_as_customer']  = 2025 - df['join_year']

df[['name', 'joined_date', 'join_year', 'join_month',
    'join_is_weekend', 'years_as_customer']]

## 4. Extracting Info from Text Columns

Useful for URL/email columns — like your phishing detector.

In [ ]:
url_df = pd.DataFrame({'url': [
    'http://paypal-login.suspicious.com/verify',
    'https://www.google.com',
    'http://192.168.1.1/bank/login',
    'https://amazon.com/orders'
]})

url_df['url_length']   = url_df['url'].str.len()
url_df['has_https']    = url_df['url'].str.startswith('https').astype(int)
url_df['num_dots']     = url_df['url'].str.count(r'\.')
url_df['has_ip']       = url_df['url'].str.contains(r'\d+\.\d+\.\d+\.\d+').astype(int)
url_df['num_slashes']  = url_df['url'].str.count('/')

url_df

> Notice: phishing URLs tend to be longer, have more dots, use http not https, and sometimes contain IP addresses. These engineered features are far more useful to a model than the raw URL string.

## 5. Interaction Features — Two Columns Combined

In [ ]:
# loan burden score — combines debt ratio and disposable income
# high debt ratio + low disposable = high risk
df['loan_burden'] = df['debt_ratio'] / (df['disposable'] + 1)  # +1 avoids division by zero

df[['name', 'debt_ratio', 'disposable', 'loan_burden', 'defaulted']].round(4)

## 6. Checking Which Features Actually Matter

After creating features, check correlation with the target. If a feature doesn't correlate — it's probably noise.

In [ ]:
# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include='number')

# Correlation with target — sorted by strength
corr = numeric_cols.corr()['defaulted'].drop('defaulted')
print("Correlation with defaulted (sorted):")
print(corr.abs().sort_values(ascending=False).round(3))

## 7. Key Rules

| Rule | Why |
|------|-----|
| Ratios beat raw numbers | Context matters — loan/income > loan alone |
| Always extract from datetime | One column → many useful features |
| Check correlation after creating | Useless features add noise |
| Think like a domain expert | What would a human look at? |
| More features ≠ better | Only keep features that help |
| Same leakage rules apply | Never use test data to create features |

> **Key insight:** Feature engineering is where domain knowledge pays off more than any algorithm choice. A well-engineered dataset with logistic regression will often beat raw data thrown at a neural network.

## Practice Task

In [ ]:
practice_data = {
    'name':        ['A', 'B', 'C', 'D', 'E', 'F'],
    'income':      [40000, 95000, 32000, 110000, 55000, 78000],
    'loan_amt':    [8000, 70000, 12000, 60000, 20000, 40000],
    'expenses':    [15000, 50000, 16000, 48000, 28000, 35000],
    'age':         [24, 48, 21, 55, 31, 42],
    'joined_date': pd.to_datetime(['2020-05-10', '2017-09-14', '2023-03-22',
                                   '2015-12-01', '2021-08-30', '2019-04-17']),
    'defaulted':   [0, 1, 0, 0, 1, 0]
}
practice_df = pd.DataFrame(practice_data)

# Q1 — Create debt_ratio = loan_amt / income
# YOUR CODE HERE

# Q2 — Create disposable = income - expenses
# YOUR CODE HERE

# Q3 — Bin age into: young (under 30), mid (30-50), senior (50+)
# YOUR CODE HERE

# Q4 — Extract year, month, and years_as_customer (2025 - year) from joined_date
# YOUR CODE HERE

# Q5 — Check corr()['defaulted'] — which new feature correlates most?
# YOUR CODE HERE